# Automatic Differentiation

In mathematics, **automatic differentiation**, simply **autodiff**, is a set of techniques to evaluate the derivative of a function. Suppose you define a function: 

$f(x,y)= x^2y+y+2$

and you need its partial derivatives (typically to perform some optimization algorithm). Your main options are manual differentiation, finite difference approximation, forward-mode autodiff, and reverse-mode autodiff

## Manual differentiation

The first approach to compute derivatives is to apply calculus to derive the appropriate equation. For the function $f(x,y)$ just defined, it is not too hard:

$\dfrac{\partial f}{\partial x} = \dfrac{\partial x^2y}{\partial x} + \dfrac{\partial y}{\partial x} + \dfrac{\partial 2}{\partial x} = y\dfrac{\partial x^2}{\partial x}+0+0=2xy$

$\dfrac{\partial f}{\partial y} = \dfrac{\partial x^2y}{\partial y} + \dfrac{\partial y}{\partial y} + \dfrac{\partial 2}{\partial y} = x^2+1+0=x^2+1$

So, the partial derivative of $f(x,y)$ at x=3 and y=4 is:

$\dfrac{\partial f}{\partial x}(3,4) = 24$  

$\dfrac{\partial f}{\partial y}(3,4) = 10$

We can also find the equations for the second order derivatives (the Hessians):

$\dfrac{\partial^2 f}{\partial x \partial x} = \dfrac{\partial (2xy)}{\partial x} = 2y$

$\dfrac{\partial^2 f}{\partial x \partial y} = \dfrac{\partial (2xy)}{\partial y} = 2x$

$\dfrac{\partial^2 f}{\partial y \partial x} = \dfrac{\partial (x^2 + 1)}{\partial x} = 2x$

$\dfrac{\partial^2 f}{\partial y \partial y} = \dfrac{\partial (x^2 + 1)}{\partial y} = 0$

so, for example

$\dfrac{\partial^2 f}{\partial x \partial x}(3,4) = 8$

$\dfrac{\partial^2 f}{\partial x \partial y}(3,4) = 6$

$\dfrac{\partial^2 f}{\partial y \partial x}(3,4) =6$

$\dfrac{\partial^2 f}{\partial y \partial y}(3,4) = 0$

This approach can become very tedious for more complex functions, and you run the risk of making mistakes. Fortunately, there are other options.

## Numerical approach

Recall the definition of derivative of a function at a point as the slope of the function at that point. More precisely: 

$\dfrac{\partial f}{\partial x} = \displaystyle{\lim_{\epsilon \to 0}}\dfrac{f(x+\epsilon, y) - f(x, y)}{\epsilon}$ 

So, if we wanted to calculate the partial derivative of $f(x,y)$ with respect to $x$  at x=3 and y=4, we could compute $f(3+\epsilon, 4)–f(3, 4)$ and divide the result by $\epsilon$, using a very small value for $epsilon$. This type of numerical approximation of the derivative is called **finite difference approximation** and this specific equation is called **Newton’s difference quotient**. That’s exactly what the following code does:

In [2]:
def gradients(func, vars_list, eps=0.0001):
    partial_derivatives = []
    base_func_eval = func(*vars_list)
    for idx in range(len(vars_list)):
        tweaked_vars = vars_list[:]
        tweaked_vars[idx] += eps
        tweaked_func_eval = func(*tweaked_vars)
        derivative = (tweaked_func_eval - base_func_eval) / eps
        partial_derivatives.append(derivative)
    return partial_derivatives

In [3]:
def f(x,y):
    return x*x*y + y + 2

def df(x, y):
    return gradients(f, [x, y])

df(3, 4)

[24.000400000048216, 10.000000000047748]

Unfortunately, the result is imprecise (and it gets worse for more complicated functions). Notice that to compute both partial derivatives, we have to call f at least three times (we called it four times in the preceding code, but it could be optimized). If there were 1,000 parameters, we would need to call f at least 1001 times. When you are dealing with large neural networks, this makes finite difference approximation way too inefficient. However, this method is so simple to implement that it is a great tool to check that the other methods are implemented correctly. 

The good news is that it is pretty easy to compute the Hessians using the numeric differentiation formula. First let's create functions that compute the first order partial derivatives (Jacobians) and then we can simply apply the gradients() function to these functions:

In [4]:
def dfdx(x, y):
    return gradients(f, [x,y])[0]

def dfdy(x, y):
    return gradients(f, [x,y])[1]

def d2f(x, y):
    return [gradients(dfdx, [3., 4.]), gradients(dfdy, [3., 4.])]

d2f(3, 4)

[[7.999999951380232, 6.000099261882497],
 [6.000099261882497, -1.4210854715202004e-06]]

## Symbolic approach

Rather than using the numerical approach, let's implement some **symbolic** techniques. For this, we will need to define classes to represent constants, variables and operations.

In [5]:
class Const(object):
    def __init__(self, value):
        self.value = value
    def evaluate(self):
        return self.value
    def __str__(self):
        return str(self.value)

class Var(object):
    def __init__(self, name, init_value=0):
        self.value = init_value
        self.name = name
    def evaluate(self):
        return self.value
    def __str__(self):
        return self.name

class BinaryOperator(object):
    def __init__(self, a, b):
        self.a = a
        self.b = b

class Add(BinaryOperator):
    def evaluate(self):
        return self.a.evaluate() + self.b.evaluate()
    def __str__(self):
        return "{} + {}".format(self.a, self.b)

class Mul(BinaryOperator):
    def evaluate(self):
        return self.a.evaluate() * self.b.evaluate()
    def __str__(self):
        return "({}) * ({})".format(self.a, self.b)

The symbolic methods are based on the chain rule. Suppose we have two functions $u$ and $v$, and we apply them sequentially to some input $x$, and we get the result $z$. So we have 

$z = v(u(x))$

which we can rewrite as 

$z = v(s)$

$s = u(x)$

now we can apply the chain rule to get the partial derivative of the output $z$ with regards to the input $x$:

$ \dfrac{\partial z}{\partial x} = \dfrac{\partial s}{\partial x} \cdot \dfrac{\partial z}{\partial s}$

now if $z$ is the output of a sequence of functions which have intermediate outputs $s_1, s_2, ..., s_n$, the chain rule still applies:

$ \dfrac{\partial z}{\partial x} = \dfrac{\partial s_1}{\partial x} \cdot \dfrac{\partial s_2}{\partial s_1} \cdot \dfrac{\partial s_3}{\partial s_2} \cdot \dots \cdot \dfrac{\partial s_{n-1}}{\partial s_{n-2}} \cdot \dfrac{\partial s_n}{\partial s_{n-1}} \cdot \dfrac{\partial z}{\partial s_n}$

In **forward mode autodiff**, the algorithm computes these terms "forward" (in the same order as the computations required to compute the output $z$), that is from left to right: 

first 

$\dfrac{\partial s_1}{\partial x}$

then 

$\dfrac{\partial s_2}{\partial s_1}$ 

and so on. 

In **reverse mode autodiff**, the algorithm computes these terms "backwards", from right to left: 

first 

$\dfrac{\partial z}{\partial s_n}$

then 

$\dfrac{\partial s_n}{\partial s_{n-1}}$

and so on.

For example, suppose you want to compute the derivative of the function 

$z(x)=\sin(x^2)$ at x=3

using forward mode autodiff. The algorithm would first compute the partial derivative 

$\dfrac{\partial s_1}{\partial x}=\dfrac{\partial x^2}{\partial x}=2x=6$

next, it would compute 

$\dfrac{\partial z}{\partial x}=\dfrac{\partial s_1}{\partial x}\cdot\dfrac{\partial z}{\partial s_1}= 6 \cdot \dfrac{\partial \sin(s_1)}{\partial s_1}=6 \cdot \cos(s_1)=6 \cdot \cos(3^2)\approx-5.46$

Now let's do the same thing using reverse mode autodiff. This time the algorithm would start from the right hand side so it would compute 

$\dfrac{\partial z}{\partial s_1} = \dfrac{\partial \sin(s_1)}{\partial s_1}=\cos(s_1)=\cos(3^2)\approx -0.91$ 

next it would compute 

$\dfrac{\partial z}{\partial x}=\dfrac{\partial s_1}{\partial x}\cdot\dfrac{\partial z}{\partial s_1} \approx \dfrac{\partial s_1}{\partial x} \cdot -0.91 = \dfrac{\partial x^2}{\partial x} \cdot -0.91=2x \cdot -0.91 = 6\cdot-0.91=-5.46$.

Let's verify this result using the gradients() function defined earlier:

In [8]:
from math import sin

def z(x):
    return sin(x**2)

gradients(z, [3])

[-5.46761419430053]

Of course both approaches give the same result (except for rounding errors), and with a single input and output they involve the same number of computations. But when there are several inputs or outputs, they can have very different performance. 

If there are many inputs, the right-most terms will be needed to compute the partial derivatives with regards to each input, so it is a good idea to compute these right-most terms first. That means using reverse-mode autodiff. This way, the right-most terms can be computed just once and used to compute all the partial derivatives. 

Conversely, if there are many outputs, forward-mode is generally preferable because the left-most terms can be computed just once to compute the partial derivatives of the different outputs. 

In Deep Learning, there are typically thousands of model parameters, meaning there are lots of inputs, but few outputs. In fact, there is generally just one output during training: the loss. This is why reverse mode autodiff is used in TensorFlow and all major Deep Learning libraries.

## Forward mode autodiff

In [7]:
Const.gradient = lambda self, var: Const(0)
Var.gradient = lambda self, var: Const(1) if self is var else Const(0)
Add.gradient = lambda self, var: Add(self.a.gradient(var), self.b.gradient(var))
Mul.gradient = lambda self, var: Add(Mul(self.a, self.b.gradient(var)), Mul(self.a.gradient(var), self.b))

dfdx = f.gradient(x)  # 2xy
dfdy = f.gradient(y)  # x² + 1

d2fdxdx = dfdx.gradient(x) # 2y
d2fdxdy = dfdx.gradient(y) # 2x
d2fdydx = dfdy.gradient(x) # 2x
d2fdydy = dfdy.gradient(y) # 0

print(dfdx.evaluate(), dfdy.evaluate())

print([[d2fdxdx.evaluate(), d2fdxdy.evaluate()],
 [d2fdydx.evaluate(), d2fdydy.evaluate()]])

AttributeError: 'function' object has no attribute 'gradient'

Note that the result is now exact, not an approximation.

## Reverse mode autodiff

In [9]:
class Const(object):
    def __init__(self, value):
        self.value = value
    def evaluate(self):
        return self.value
    def backpropagate(self, gradient):
        pass
    def __str__(self):
        return str(self.value)

class Var(object):
    def __init__(self, name, init_value=0):
        self.value = init_value
        self.name = name
        self.gradient = 0
    def evaluate(self):
        return self.value
    def backpropagate(self, gradient):
        self.gradient += gradient
    def __str__(self):
        return self.name

class BinaryOperator(object):
    def __init__(self, a, b):
        self.a = a
        self.b = b

class Add(BinaryOperator):
    def evaluate(self):
        self.value = self.a.evaluate() + self.b.evaluate()
        return self.value
    def backpropagate(self, gradient):
        self.a.backpropagate(gradient)
        self.b.backpropagate(gradient)
    def __str__(self):
        return "{} + {}".format(self.a, self.b)

class Mul(BinaryOperator):
    def evaluate(self):
        self.value = self.a.evaluate() * self.b.evaluate()
        return self.value
    def backpropagate(self, gradient):
        self.a.backpropagate(gradient * self.b.value)
        self.b.backpropagate(gradient * self.a.value)
    def __str__(self):
        return "({}) * ({})".format(self.a, self.b)
    
x = Var("x", init_value=3)
y = Var("y", init_value=4)
f = Add(Mul(Mul(x, x), y), Add(y, Const(2))) # f(x,y) = x²y + y + 2

result = f.evaluate()
f.backpropagate(1.0)

print(f)
print(result)
print(x.gradient)
print(y.gradient)

((x) * (x)) * (y) + y + 2
42
24.0
10.0


In this implementation the outputs are just numbers, not symbolic expressions, so we are limited to first order derivatives. However, we could have made the `backpropagate()` methods return symbolic expressions rather than values (e.g., return `Add(2,3)` rather than 5). This would make it possible to compute second order gradients (and beyond). This is what TensorFlow does, as do all the major libraries that implement autodiff:

In [10]:
import tensorflow as tf
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

x = tf.Variable(3.)
y = tf.Variable(4.)

with tf.GradientTape() as tape:
    f = x*x*y + y + 2
    df_dx, df_dy = tape.gradient(f, [x, y])

with tf.GradientTape(persistent=True) as tape:
    f = x*x*y + y + 2
    df_dx, df_dy = tape.gradient(f, [x, y])

d2f_d2x, d2f_dydx = tape.gradient(df_dx, [x, y])
d2f_dxdy, d2f_d2y = tape.gradient(df_dy, [x, y])

hessians = [[d2f_d2x, d2f_dydx], [d2f_dxdy, d2f_d2y]]
print(hessians)

2023-06-14 17:10:37.546419: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[[<tf.Tensor: shape=(), dtype=float32, numpy=8.0>, <tf.Tensor: shape=(), dtype=float32, numpy=6.0>], [<tf.Tensor: shape=(), dtype=float32, numpy=6.0>, None]]
